In [1]:
import os

if "notebooks" in os.getcwd():
	os.chdir("..")

import glob
import numpy as np 
import re 

from scipy.ndimage import rotate
from scipy import ndimage as ndi

from skimage import io, filters, morphology

import matplotlib.pyplot as plt

%matplotlib qt

from numpy2stl.numpy2stl import numpy2stl, triangles_to_facets, writeSTL, polygon
from numpy2stl.numpy2stl import Solid

from geo2stl.geo2stl import *
from geo2stl import geo2stl as g2s
from geo2stl import sat2stl as s2s

from city2stl import dem2stl as d2s

import notebooks.figure as figs 

import trimesh
#trimesh.interfaces.blender._blender_executable = r"C:\Program Files\Blender Foundation\Blender 4.1\blender.exe"


In [2]:
import pymeshlab as ml
import trimesh
import numpy as np

from numpy2stl.numpy2stl.simplify import simplify_mesh_surfaces
from numpy2stl.numpy2stl.save import writeOBJ
from numpy2stl.numpy2stl import puzzle, view, verify, simplify

from numpy2stl.numpy2stl import solid

In [3]:
%load_ext autoreload
%autoreload 2

## Union, difference and intersection

In [ ]:
width = im.shape
b = 200
m = 50        # Distance from corners to start the notch
base_n = 10   # The depth of the notch

pts_list = puzzle.make_puzzle_pts(width, b, m, base_n)

plt.figure()
for pts in pts_list:
	plt.plot(pts[:, 0], pts[:, 1], '-')
	plt.fill(pts[:, 0], pts[:, 1], alpha = 0.3)
plt.grid(True)
plt.axis("equal")

#### Get data from source and place in root
https://www.gebco.net/data-products/gridded-bathymetry-data/gebco-2020

""

In [4]:
Ocean_root = "C:/Users/eac84/Desktop/Desktop/Tasks/srtm_tifs/gebco_2020_geotiff/"
tile_files = glob.glob(Ocean_root+"*.tif")
out_dir = "C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files/"
dem_root = "C:/Users/eac84/Desktop/Desktop/FILES"

# North America

## Wiss

In [ ]:
(N, S, E, W) =  40.085,40.010, -75.182,   -75.235

In [ ]:
bounds = (W, S, E, N)

dem = d2s.DEM(root=dem_root, geo_bounds=(N,S,E,W))
im = dem.data.copy()

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))


im = rescale(im, max_size=600, height=20, base=0.1)


figs.plot_data(im)

In [ ]:
name = "Wiss"
savefile(out_dir, name, im+1)

## Applelachans 

In [ ]:
(N, S, E, W) =  48, 33,  -64, -88


In [ ]:
target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()
im[im<0] = im[im<0]*0.1
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05

#im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = rescale(im, max_size=900, height=25, base=0.1)

im2 = im.copy()
remap = np.array([[0,0],[3,5],[7.5,6],[8,8.5],[9,9],[11,15],[15,21],[17,23],[30,30]])
im2[:,:,] = np.interp(im[:,:,],remap[:,0], remap[:,1])

figs.plot_data(im2)
fig , axs = plt.subplots(2,1, figsize=(12, 6), 
				layout = 'tight', 
				sharex=True, sharey=True)

axs = axs.ravel()
axs[0].hist(im.ravel()[::10],100)
axs[1].hist(im2.ravel()[::10],100)

fig , axs = plt.subplots(1,1, figsize=(12, 6), 
				layout = 'tight', 
				sharex=True, sharey=True)
axs.plot(remap[:,0],remap[:,1],"-o")
axs.set_aspect("equal")
print("")


In [ ]:
figs.plot_data(im)

In [ ]:
name = "Appalachia"
savefile(out_dir, name, im)

## NORTH JERSEY

In [8]:
from notebooks.experimental import get_landspace_model, resize_max, rescale

In [9]:
(N, S, E, W) =  41.5, 40,  -73.5, -75.5
(N, S, E, W) =  41.5, 40.25,  -73.75, -75.25

In [10]:
dim = 600

target_bbox = np.array((N, S, E, W))
N,S,E,W = (N*1. , S*1., E*1., W*1.)
bounds_NW = (N,S), (W,E) 

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = proj_map_geo_to_2D(result,np.array((N, S, E, W)))

im = result.copy()
im[im<0] = im[im<0] - np.ptp(im.ravel())*0.01

if 0:

	sat = s2s.fetch_bbox_image( N,S,E,W, scale=100, dataset="esa")
	water = 20.0*(sat == 80)

	im = resize_max(im, max_size=dim)
	water = resize_max(water, max_size=dim)

	water = water.clip(0,10)*2
	im = im - water
	
im = rescale(im, max_size=dim, height=25, base=0.2, clip=None)

x = np.array(target_bbox)[np.array([3,2,1,0])]
figs.plot_data(im, bbox = x )
plt.figure()
plt.imshow(water)


==== Stitching tiles without rasterio ====
Target bounding box: [ 41.5   40.25 -73.75 -75.25]
✅ Finished stitching without rasterio.
--


NameError: name 'water' is not defined

In [9]:
#im2 = scipy.ndimage.median_filter(im, 3)
models = {}
vertices, faces = get_landspace_model(im, bounds_NW, 500, simplify=False)
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.fix_normals()
models["landscape"] = mesh.vertices, mesh.faces


Creating top...
Creating walls...
Creating bottom...


In [11]:

x = 20*((im/10)+1).round(0)[::2,::2]
vertices, faces = get_landspace_model(x, bounds_NW, 500, simplify=False)
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.fix_normals()
vx,fs  = mesh.vertices, mesh.faces

model_1 = {"Before":(vx, fs)}

verify.check_model_status( model_1 )


Creating top...
Creating walls...
Creating bottom...
ID: Before
Centroid (XYZ):  [149500.   124500.    22773.47]
Dimensions:      [299000. 249000.  40000.]
Watertight:      ✅ Yes
Valid Volume:    ✅ Yes
------------------------------


In [ ]:
plt.close("all")
fs2 = simplify_mesh_surfaces(*model_1["Before"])
##########################
mesh = trimesh.Trimesh(vertices=vx, faces=fs2)
mesh.fix_normals()
vx2,fs2  = mesh.vertices, mesh.faces
model = {"After":(vx2, fs2)}

verify.check_model_status( model )

print(len(faces), len(fs2))
view.render_models_napari( model )


<string>:65: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.


In [15]:
plt.close("all")

In [38]:
view.render_models_napari( model )

opening - Napari


Invalid schema for package 'ome-types', please run 'npe2 validate ome-types' to check for manifest errors.


After
22540


In [35]:
fs2 = simplify_mesh_surfaces(*model_1["Before"])

In [11]:
%load_ext line_profiler

In [25]:
simplify.simplify_mesh_surfaces(vx,fs)

array([[   53,    55,   528],
       [  528,   781,   780],
       [  779,  1031,  1030],
       ...,
       [  381,   692,   383],
       [  379,   691,   381],
       [30548, 30801, 30549]], dtype=int64)

In [23]:
%lprun -f simplify.simplify_mesh_surfaces simplify.simplify_mesh_surfaces(vx,fs)

Timer unit: 1e-07 s

Total time: 0.896316 s
File: c:\Users\eac84\OneDrive\Documents\Projects\3D Maps\Code\strm2stl\numpy2stl\numpy2stl\simplify.py
Function: simplify_mesh_surfaces at line 16

Line #      Hits         Time  Per Hit   % Time  Line Contents
    16                                           def simplify_mesh_surfaces(vertices, faces, min_faces=10):
    17                                           
    18         1    2833857.0    3e+06     31.6  		surfaces, normals = get_surfaces(vertices, faces, normals=None)
    19                                           
    20         1          9.0      9.0      0.0  		faces_out = []
    21      9549      44874.0      4.7      0.5  		for f, sub_faces in enumerate(surfaces):
    22                                           
    23      9548     412230.0     43.2      4.6  				face_list = faces[sub_faces]	
    24      9548      41277.0      4.3      0.5  				if len(sub_faces)<min_faces: 
    25      9487      40550.0      4.3      0.5 

In [ ]:
#for idx in range(20000,30000):

from shapely.geometry import Polygon
from shapely import constrained_delaunay_triangles

In [ ]:
plt.close("all")

plt.figure()
tris = vert2[faces]
#tris = vx[faces]
for t in tris:
	plt.fill(t[:,0], t[:,1], alpha=0.3)

for p in perimeters:
	p_vx = vx[p]
	plt.plot(p_vx[:,0], p_vx[:,1], "k")

plt.figure()
for t in tris2:
	plt.fill(t[0], t[1], alpha=0.3)

In [ ]:
_, fs2 = simplify_mesh_surfaces(*models_land["landscape"])

verify.check_model_status({"test":(vx,fs2)} )
print(len(models_land["landscape"][1]))
print(len(fs2))

In [ ]:
render_models_napari( {"test":(vx,fs2)})

In [ ]:
width = im.shape
b = 200
m = 50        # Distance from corners to start the notch
base_n = 10  
puzzle = puzzle.make_puzzle_model(width, b, m, base_n)

model = models["landscape"]
puzzle_cut = puzzle.cut_puzzle_pieces(model, puzzle)

view.render_models_napari(puzzle_cut)

verify.check_model_status(puzzle_cut)
writeOBJ("test.obj", puzzle_cut)


In [ ]:
#_,new_faces = simplify.simplify_surface(projected[:,:2], perimeters)

In [ ]:
name = "NorthNJ"
write.savefile(out_dir, name, im)

## Penn

In [ ]:
(N, S, E, W) =  42.8, 39.5,  -73.75, -80.


In [ ]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = result.copy()

im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05

im = rescale(im, max_size=800, height=35, base=0.1,
         clip=None , smooth=[.1,99.95])

In [ ]:
name = "Penn"
write.savefile(out_dir, name, im)

In [ ]:
from geo2stl import write

## Great Lakes

In [ ]:
(N, S, E, W) =  49.677, 40.465,  -74, -93.387
#(N, S, E, W) =  49.677, 41,  -70, -94
(N, S, E, W) =  49.5, 41,  -74, -94

In [ ]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)


im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]
im = im * 0.1

im = rescale(im, max_size=800, height=25, base=0.1,
         clip=[.01,99.99], smooth=3)



In [ ]:
figs.plot_data(im)

In [ ]:
name = "GreatLakes_"
savefile(out_dir, name, im)

In [ ]:
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.show()

In [ ]:

# Simplify mesh to 10% of original face count

mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
simplified = mesh.simplify_quadric_decimation(0.9)
vertices = simplified.vertices.copy().astype(float)
faces = simplified.faces
#


In [ ]:
simplified.export(out_dir + "simplified.stl")

## INDIAN RIVER

In [ ]:
N,S,E,W = 28,27,-80,-80.6

In [ ]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im[im>0] = im[im>0] + 10
im[im<-20] = -20



## Colorado

In [ ]:
N,S,E,W = [  41.0034002,   36.9925245, 
       	   -102.041585 , -109.0601879]


In [ ]:
target_bbox = np.array((N, S, E, W))
NSEW = np.array([N,S,E,W])

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

x = 1500
im[im>x] = 0.3*(im[im>x]-x)+x

im = im * 0.02

im = resize_geo_aspect(im, NSEW)

filt = ndi.gaussian_filter(im, sigma=50)
im = im - filt*0.7

im = rescale(im, max_size=800, height=50, base=0.1,
         clip=[.01,99.99], smooth=3)



In [ ]:
name = "Colorado1"
savefile(out_dir, name, im)

# Central America

## Caribbean

In [ ]:
N,S,E,W = 31,6,-54,-102

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im>0] = im[im>0]+1000

#im3 = proj_map_height(im2, np.array([n,s,e,w]))
im = proj_map_geo_to_2D(im, target_bbox )

im = rescale(im, max_size=1000, height=30, base=0.1,
         clip=[.01,99.99], smooth=None)


In [ ]:
tri = np3D.numpy2stl(im4)
vertices, faces = np3D.vertices_to_index(tri)
mesh = trimesh.Trimesh(vertices,faces)
mesh.show()

In [ ]:
facets = np3D.triangles_to_facets(tri)
np3D.writeSTL(facets, "Ocean_flat.stl")

## Puerto Rico

In [ ]:
N,S,E,W  = 18.75, 17.65, -65.15, -67.35

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.2
im[im>0] = im[im>0] + 100

im_x = im - im.min() + (im.ptp()*0.1)

scale = 2
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02


In [ ]:
name = "PuertoRico"
savefile(out_dir, name, im_x)

## Keys

In [ ]:
(N, S, E, W) = 25.9,24.3, -80,-82.0

In [ ]:
target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<-100] = -100
im[im<0] = im[im<0]*0.1
im[im>0] = im[im>0] + 10

#im[im<0] = im[im<0]-200

scale = 2
im_x = im.copy()
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02

DEM = im_x
rotation = 55 - 90

DEM = rotate(DEM,rotation, reshape=True, cval=-10)

DEM = DEM[480:820,:]
DEM[DEM==-10] = DEM[DEM>-10].min()




## Bahamas

In [ ]:
N,S,E,W = 35,10,-60,-88


In [ ]:

target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()

im = proj_map_geo_to_2D(im, target_bbox , clip_out=False)
im[np.isnan(im)] = -10000

DEM = im.copy()
lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

#DEM = DEM[::10,::10]
rotation = 38

DEM = rotate(DEM,rotation, reshape=True, cval=-0)

x1,x2,y1,y2 = 3800,6100, 2200,6700

DEM2 = DEM[x1:x2, y1:y2].copy()

lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

DEM2[DEM2>0] = DEM2[DEM2>0]+np.ptp(DEM2)*0.1
DEM2 = resize_max(DEM2, max_size=1000)
DEM2 = DEM2 - DEM2.min() + DEM2.ptp()*0.1
DEM2 = DEM2 / DEM2.ptp() * 25

plt.close("all")	
plt.figure()
plt.imshow(DEM,cmap="jet")
plt.plot([y1,y1,y2,y2,y1],[x1,x2,x2,x1,x1], color="white", linewidth=3)
plt.grid(True)

plt.figure()
plt.imshow(DEM2,cmap="jet")
plt.grid(True)


plt.figure()
_ = plt.hist(DEM2.ravel(), bins=100)

In [ ]:
name = "Antillas_38deg"
savefile(out_dir, name, DEM2)

## Mexico

In [ ]:
N,S,E,W = 37,11,-75,-122

target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

In [ ]:
im = result.copy()

im = resize_max(im, max_size=1000)
im = proj_map_geo_to_2D(im, target_bbox )


im[im<0] = im[im<0] * 0.05
im[im<0] = im[im<0] - im.ptp()*0.1
im = im / im.ptp()*30
im  = im - im.min() + im.ptp()*0.1

figs.plot_data(im)


# South America

## Colombia

In [ ]:
from city2stl import osm2stl as o2s
from city2stl import create 

#from sd2s import embed_lines
from skimage import morphology
from skimage.draw import line_aa, polygon2mask

In [ ]:
N,S,E,W = 14,-6,-64,-84
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

In [ ]:

nsew = np.array([N,S,E,W])

x1 = 300 
x3 = 2000
im_ = im.copy()
im_[im_>x1] = im_[im_>x1] + (x3-x1)
bx = (im_<=x1) & (im_>0)
im_[bx] = im_[bx] * (x3/x1)

im_[im_<0] = im_[im_<0]*0.5
im_[im_>0] = im_[im_>0] + 10

#############
scale = 0.5
sz = tuple((np.array(im_.shape)[[1,0]]*scale).astype(int))
im_ = im_ * scale
im_ = cv2.resize(im_,sz)
im_ = im_ * 0.01

###############
sealine = im_>0
sealine = morphology.binary_dilation(sealine, morphology.disk(2)) & ~sealine
im_ = im_ - (sealine* im_.ptp()*0.01)

################
im_lines = get_border(nsew, "Colombia", im_.shape)
im_lines[im_<0] = 0 
im_ = im_ - (im_lines* im_.ptp()*0.01)

################
im_ = im_ - np.min(im_) + (im_.ptp()*.01)

In [ ]:

name = "Colombia"
savefile(out_dir, name, im_ )

## Medellin

## Brazil

In [ ]:
im_all = io.imread(tile_files[1])
plt.imshow(im_all[::10,::10])

In [ ]:
index_list = [
[5500,5800,10200,10700],
[5420,5550,11150,11350],
[5200,5800,10200,11500],
[4000,7000,7500,10500],
[5800,6500,8000,9000],
[5900,6200,8300,8800,]
]

plt.close("all")
plt.figure()
plt.imshow(im_all, cmap="jet")
for index in index_list:
	plt.plot(
		[index[2],index[2],index[3],index[3],index[2]],
		[index[0],index[1],index[1],index[0],index[0]], color="white", linewidth=3)
#plt.grid(True)


for index in index_list:
	imx = im_all[index[0]:index[1],index[2]:index[3]]*1.
	imx = normalize(imx)
	plt.figure()
	plt.imshow(imx, cmap="jet")

## Amazon

In [ ]:
s2s.initialize_earth_engine()
N,S,E,W = 14,-20,-45,-85


In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc", scale=2000)

im = result.copy()
im = subtract_water(im, img, height = 0.2, ocean_level=1)

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = rescale(im, max_size=600, height=30, base=0.2, clip= [.01,99.99], smooth=3)


figs.plot_data(im)

plt.figure()
img2 = resize_max(img*1., max_size=600)
plt.imshow(img2, cmap="jet")

name = "Amazon"
#savefile(out_dir, name, im)

# Europe

## Norway

In [ ]:
N=72.07
S=57.03
E= 33.28 
W= -3

In [ ]:
target_bbox = (N, S, E, W)

result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=800)

## North Sea

In [ ]:
N, S, E, W =67, 53, 38, -2

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

if 1:
	im[im<0] = np.sqrt(np.abs(im[im<0])) * np.sign(im[im<0])
	im[im<0] = im[im<0] / (np.ptp(im[im<0])) * result[im<0].ptp()*0.5
	
im[im>0] = im[im>0] + np.ptp(im.ravel())*0.05

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), clip_out=True)
im = rescale(im, max_size=800, height=25, base=0.1, clip= [.01,99.99], smooth=3)

In [ ]:
name = "NorthSea"
savefile(out_dir, name, im)

## IBERIA

In [ ]:
N,S,E,W = 44.4, 34.8, 5.3,-11

In [ ]:
target_bbox = np.array((N, S, E, W))
nsew = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.25
im[im>0] = im[im>0]+100

im_r = im.copy()
im_r[im_r<0]=0
im_r = (ndi.gaussian_filter(im_r,10) - im_r).clip(0)
im = im - 2*im_r


im2 = proj_map_geo_to_2D(im, np.array(nsew))
im2 = im2 - np.min(im2) + 1


scale = 0.3
sz = tuple((np.array(im2.shape)[[1,0]]*scale).astype(int))
im2 = im2 * scale
im2 = cv2.resize(im2,sz)
im2 = im2 * 0.02


In [ ]:
figs.plot_data(im2)

In [ ]:
name = "Ibera"
savefile(out_dir, name, im2 )


## Mediterrian 

In [ ]:
N, S, W, E =  48.5, 28.5, -15, 45.2

In [ ]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), True)
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [5,99.99])
im = im.clip(lo,hi)

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0]-im.ptp()*0.05

im = im / im.ptp()*30
im  = im - im.min() + im.ptp()*0.05


In [ ]:
figs.plot_data(im)

In [ ]:
view3D_napari(im)

In [ ]:
name = "Mediterranean"
savefile(out_dir, name, im)

# AFRICA

## Pan Africa

In [ ]:
index = [11000,31500,15500,36000]


x = [0,21600*2]
y = [-90.0,90.0]

coor = np.interp(index, x,y)
coor[:2] = -coor[:2]
print(coor)

In [ ]:
np.array(tile_files)[[1,5,6,2]]

In [ ]:
im_4 = io.imread(tile_files[2])
im_1 = io.imread(tile_files[5])
im_2 = io.imread(tile_files[6])
im_3 = io.imread(tile_files[1])

im_all = np.vstack([np.hstack([im_1,im_2]),np.hstack([im_3,im_4])])
im_0 = im_all[index[0]:index[1],index[2]:index[3]] * 1.0



In [ ]:

target_bbox = coor[[0,1,3,2]]
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)


In [ ]:
plt.figure()
plt.imshow(result[::10,::10]/(result.max()))
plt.figure()
plt.imshow(im_0[::10,::10]/(im_0.max()))

In [ ]:
im_0.shape, result.shape

In [ ]:
im_x = im_0
sz = tuple((np.array(im_x.shape)[[1,0]]*0.02).astype(int))
im_x = im_x * 0.1
im_x = cv2.resize(im_x,sz)*1.0

sz = tuple((np.array(im_x.shape)[[1,0]]*2).astype(int))
im_x = cv2.resize(im_x,sz)


im_x[im_x<0] = im_x[im_x<0]*.1
im_x[im_x<0] = im_x[im_x<0]*.2

x = 200
im_x[im_x>x] = 0.2*(im_x[im_x>x]-x)+x

x = 100
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = 50
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = -10
im_x[im_x<x] = 5*(im_x[im_x<x]-x)+x



im_x[im_x<0] = im_x[im_x<0] - 0.05*(im_x.max()-im_x.min())

n1,s1,e1,w1 = mat2coor( [0, 180, -90,90], [43200, 43200], index)
s1 = s1 - 180

nsew1 = (n1,s1,e1,w1)

im_x2 = proj_map_geo_to_2D(im_x,np.array(nsew1))

im_x = im_x - im_x.min() + 10

In [ ]:

name = "Africa"
savefile(out_dir, name, im_x)

# Asia 

## Bengal

In [ ]:
s2s.initialize_earth_engine()

In [ ]:
N,S,E,W =  29, 20 ,95, 84

In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc")

im = result.copy()
img2 = img.copy().astype(np.uint8)

im = resize_max(im, max_size=1200)
img2 = filters.median(img2, np.ones((3,3)))
img2 = cv2.resize(img2, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_LINEAR).astype(int)
img2[im<0] = 200
img2[im>500] = 0
img2 = img2 / 100
img2 = (img2).clip(0,1)


x = 1
xmax = im.max()
im[im>x] = ((im[im>x]/im.max())**0.7 )*im.max()


im = im  - img2*np.ptp(im.ravel())*0.1

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=600)
im = filters.median(im, np.ones((3,3)))
im = im.clip(*np.percentile(im.ravel(), [.01,99.99]))
im  = im - im.min() + im.ptp()*0.2
im = im / im.ptp() * 30




In [ ]:
name = "Bengal"
savefile(out_dir, name, im)

## Persia

In [ ]:
N,S,E,W = 41,  24 ,65, 41


In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))


im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im  = im - im.min() + im.ptp()*0.1

im = resize_max(im, max_size=900)
im = filters.median(im, np.ones((3,3)))
lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)




In [ ]:
name = "Persia"
savefile(out_dir, name, im)

## Japans

In [ ]:
N,S,E,W = 55.066, 28.75, 153.69, 118
N,S,E,W = 48, 30, 150, 122


In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im[im<0] = im[im<0].clip(-4000)

im = resize_max(im, max_size=800)
im = filters.median(im, np.ones((3,3)))

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30

In [ ]:
figs.plot_data(im)

In [ ]:
name = "Japans"
savefile(out_dir, name, im)

## South China Sea

In [ ]:
N,S,E,W = 25.938, -11.716, 133.769, 91.58
N,S,E,W = 25.938, -11.716, 136, 88
N,S,E,W = 29, -14, 140 , 86

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)

im = filters.median(im, np.ones((3,3)))

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0] - im.ptp()*0.05

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30


In [ ]:

name = "SouthChinaSea"
savefile(out_dir, name, im)